In [1]:
import pandas as pd
import numpy as np
import scipy.stats as st

In [2]:
df = pd.read_csv("steam_indie_merged_data.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'steam_indie_merged_data.csv'

In [ ]:
# 1. 데이터 불러오기 (DB에서 불러온 df가 있다고 가정)
# df = pd.read_csv('steam_indie_merged_data.csv')

# 날짜 변환 및 총 리뷰 수 계산
df['release_date_dt'] = pd.to_datetime(df['release_date'], errors='coerce')
df['total_reviews'] = df['positive'] + df['negative']

# 2. 기본 필터링 (2024년 이후, 캐주얼 장르)
filtered_df = df[
    (df['release_date_dt'] >= '2024-01-01') & 
    (df['genres'].str.contains('Casual', case=False, na=False))
].copy()

# ==========================================
# 3. 핵심: 리뷰 수 기준 '규모별 계층화' 적용
# ==========================================
def categorize_size(reviews):
    if reviews >= 501:
        return "3. 대형 (501건 이상)"
    elif reviews >= 101:
        return "2. 중형 (101~500건)"
    elif reviews >= 30:
        return "1. 소형 (30~100건)"
    else:
        return "0. 분석 제외 (30건 미만)"

filtered_df['size_tier'] = filtered_df['total_reviews'].apply(categorize_size)

# 분석 대상만 필터링 (30건 미만 제외)
analysis_df = filtered_df[filtered_df['size_tier'] != "0. 분석 제외 (30건 미만)"].copy()

# ==========================================
# 4. 윌슨 스코어 및 긍정률 계산
# ==========================================
def calculate_wilson_score(positive, total, confidence=0.95):
    if total == 0: return 0.0
    z = st.norm.ppf(1 - (1 - confidence) / 2)
    p_hat = positive / total
    numerator = p_hat + z**2 / (2 * total) - z * np.sqrt((p_hat * (1 - p_hat) + z**2 / (4 * total)) / total)
    denominator = 1 + z**2 / total
    return numerator / denominator

analysis_df['positive_rate(%)'] = (analysis_df['positive'] / analysis_df['total_reviews']) * 100
analysis_df['wilson_score'] = analysis_df.apply(
    lambda row: calculate_wilson_score(row['positive'], row['total_reviews']), axis=1
)

# ==========================================
# 5. 규모(체급)별 상/하위 3개 게임 추출
# ==========================================

# 규모별 최상위 게임
top_by_size = analysis_df.sort_values(
    by=['size_tier', 'wilson_score'], ascending=[True, False]
).groupby('size_tier').head(5)

# 규모별 최하위 게임
bottom_by_size = analysis_df.sort_values(
    by=['size_tier', 'wilson_score'], ascending=[True, True]
).groupby('size_tier').head(5)

# ==========================================
# 결과 출력
# ==========================================
columns_to_show = ['name_store', 'size_tier', 'total_reviews', 'positive_rate(%)', 'wilson_score']

print(" [규모별 상위 3개 게임 (소형/중형/대형)]")
display(top_by_size[columns_to_show])

print("\n [규모별 하위 3개 게임 (소형/중형/대형)]")
display(bottom_by_size[columns_to_show])

 [규모별 상위 3개 게임 (소형/중형/대형)]


,name_store,size_tier,total_reviews,positive_rate(%),wilson_score
9081,This Magical Girl is a B☆tch,1. 소형 (30~100건),96,100.000000,0.961524
37193,Cape Hideous,1. 소형 (30~100건),94,100.000000,0.960738
20750,WYRMHALL: Brush and Banter,1. 소형 (30~100건),87,100.000000,0.957712
46116,Projected Dreams,1. 소형 (30~100건),86,100.000000,0.957242
31741,Hanako's Flower Shop,1. 소형 (30~100건),83,100.000000,0.955765
8266,A Webbing Journey,2. 중형 (101~500건),497,99.798793,0.988692
16153,"Shooters, Ready!",2. 중형 (101~500건),229,100.000000,0.983502
29757,Desvelado,2. 중형 (101~500건),320,99.687500,0.982514
28079,Wilmot Works It Out,2. 중형 (101~500건),466,99.356223,0.981245
46121,Little Adventurer Treasure Hunt,2. 중형 (101~500건),152,100.000000,0.975350



 [규모별 하위 3개 게임 (소형/중형/대형)]


,name_store,size_tier,total_reviews,positive_rate(%),wilson_score
42678,Crab Game 2,1. 소형 (30~100건),35,22.857143,0.120659
53949,My Museum: Treasure Hunter,1. 소형 (30~100건),35,22.857143,0.120659
11042,Banana-Shaped,1. 소형 (30~100건),46,28.260870,0.173242
34404,Wizlite: Everybody loved RPGs,1. 소형 (30~100건),33,30.303030,0.173755
13749,Obscula,1. 소형 (30~100건),37,29.729730,0.174895
33969,我亲爱的妹妹 Happy Sisters Life,2. 중형 (101~500건),117,12.820513,0.079256
40869,Cinemaster Cinema Simulator,2. 중형 (101~500건),126,42.857143,0.345538
25417,Forest Ranger Simulator,2. 중형 (101~500건),226,42.477876,0.362119
10431,Boldly Forward,2. 중형 (101~500건),145,47.586207,0.396250
25639,Star Command Galaxies,2. 중형 (101~500건),484,44.008264,0.396503


In [ ]:
import pandas as pd
import glob
import os

# 1. 폴더 경로 지정 (에러 메시지에 나온 경로)
folder_path = './Game Reviews'

# 2. 해당 폴더 내의 모든 CSV 파일 경로 찾기 (*.csv 로 수정됨)
file_list = glob.glob(os.path.join(folder_path, '*.csv'))

print(f"총 {len(file_list)}개의 CSV 파일을 찾았습니다. 병합을 시작합니다...")

# 3. CSV 파일들을 하나씩 읽어서 리스트에 담기
df_list = []
for file in file_list:
    try:
        # 🌟 핵심 수정: read_excel 대신 read_csv 사용! 🌟
        temp_df = pd.read_csv(file)
        df_list.append(temp_df)
    except Exception as e:
        print(f"⚠️ 에러 발생 파일: {file} - {e}")

# 4. 리스트에 담긴 데이터를 하나의 거대한 데이터프레임으로 합치기
if df_list:
    merged_df = pd.concat(df_list, ignore_index=True)
    print("\n✅ 모든 CSV 파일 병합 성공!")
    print(f"합쳐진 전체 리뷰 수: {len(merged_df):,}개")
    
    # post_time 결측치 확인
    if 'post_time' in merged_df.columns:
        missing_post_time = merged_df['post_time'].isnull().sum()
        total_rows = len(merged_df)
        missing_ratio = (missing_post_time / total_rows) * 100
        print(f"\n🔍 'post_time' 빈칸(결측치) 수: {missing_post_time:,}개 (전체의 {missing_ratio:.2f}%)")
    else:
        print("\n⚠️ 합쳐진 데이터에 'post_time' 컬럼이 존재하지 않습니다.")
        
else:
    print("경로 내에 합칠 수 있는 CSV 파일이 없습니다.")

총 23107개의 CSV 파일을 찾았습니다. 병합을 시작합니다...

✅ 모든 CSV 파일 병합 성공!
합쳐진 전체 리뷰 수: 31,832,390개

⚠️ 합쳐진 데이터에 'post_time' 컬럼이 존재하지 않습니다.


In [ ]:
# 1. df2의 전체 컬럼명 출력
print("--- df2 전체 컬럼명 ---")
for i, col in enumerate(merged_df.columns):
    print(f"{i+1}. {col}")

print("\n--- df2 데이터 요약 정보 ---")
# 2. 각 컬럼의 데이터 타입과 총 데이터 개수, 결측치 여부 확인
merged_df.info()

print("\n--- df2 데이터 미리보기 (상위 3줄) ---")
# 3. 실제 데이터가 어떻게 생겼는지 눈으로 확인
display(merged_df.head(3))

--- df2 전체 컬럼명 ---
1. user
2. playtime
3. post_date
4. helpfulness
5. review
6. recommend
7. early_access_review

--- df2 데이터 요약 정보 ---
<class 'pandas.DataFrame'>
RangeIndex: 31832390 entries, 0 to 31832389
Data columns (total 7 columns):
 #   Column               Dtype  
---  ------               -----  
 0   user                 str    
 1   playtime             float64
 2   post_date            str    
 3   helpfulness          int64  
 4   review               str    
 5   recommend            str    
 6   early_access_review  object 
dtypes: float64(1), int64(1), object(1), str(4)
memory usage: 1.7+ GB

--- df2 데이터 미리보기 (상위 3줄) ---


,user,playtime,post_date,helpfulness,review,recommend,early_access_review
0,Kutar'k,9.5,"May 16, 2021",13,"멸망한 인류가 간직한 강력한 힘의 해방.스플릿 불렛(Split Bullet), 벡토...",Recommended,NaN
1,DogelordCS,7.7,"July 5, 2021",5,"After beating this game, I would not recommend...",Not Recommended,NaN
2,점박이뱀,14.5,"May 16, 2021",8,형형색색 삐까번쩍 심플이즈베스트 도형디자인이 돋보이는 2D K-탄막슈팅 [어센션]입...,Recommended,NaN


In [ ]:
import pandas as pd
import glob
import os

folder_path = './Game Reviews'
file_list = glob.glob(os.path.join(folder_path, '*.csv'))

df_list = []
for file in file_list:
    try:
        temp_df = pd.read_csv(file)
        
        # 🌟 핵심 수정: 파일명(예: 1001270_4904.csv)에서 앞의 게임 ID만 추출
        file_name = os.path.basename(file).replace('.csv', '')  # '1001270_4904'
        app_id = file_name.split('_')[0]                        # '1001270' (언더바 앞부분)
        
        temp_df['appid'] = app_id 
        
        df_list.append(temp_df)
    except Exception as e:
        print(f"⚠️ 읽기 실패: {file} - {e}")

# df2로 최종 병합 및 CSV 저장
if df_list:
    df2 = pd.concat(df_list, ignore_index=True)
    print(f"✅ 병합 완료! 총 데이터 수: {len(df2):,}개")

    # 곧바로 CSV 파일로 저장
    df2.to_csv('merged_steam_reviews.csv', index=False, encoding='utf-8-sig')
    print("✅ 파일 저장 완료: merged_steam_reviews.csv\n")

    # 결과 데이터 미리보기 (appid가 제대로 들어갔는지 확인용)
    display(df2[['appid', 'user', 'playtime', 'post_date', 'recommend']].head())
    
else:
    print("경로 내에 합칠 수 있는 CSV 파일이 없습니다.")

✅ 병합 완료! 총 데이터 수: 31,832,390개
✅ 파일 저장 완료: merged_steam_reviews.csv



,appid,user,playtime,post_date,recommend
0,1000000,Kutar'k,9.5,"May 16, 2021",Recommended
1,1000000,DogelordCS,7.7,"July 5, 2021",Not Recommended
2,1000000,점박이뱀,14.5,"May 16, 2021",Recommended
3,1000000,haramey,8.1,"June 28, 2021",Not Recommended
4,1000000,boymin,15.4,"May 18, 2021",Recommended


In [ ]:
# 1. df2의 전체 컬럼명 출력
print("--- df2 전체 컬럼명 ---")
for i, col in enumerate(df2.columns):
    print(f"{i+1}. {col}")

print("\n--- df2 데이터 요약 정보 ---")
# 2. 각 컬럼의 데이터 타입과 총 데이터 개수, 결측치 여부 확인
df2.info()

print("\n--- df2 데이터 미리보기 (상위 3줄) ---")
# 3. 실제 데이터가 어떻게 생겼는지 눈으로 확인
display(df2.head(30))

--- df2 전체 컬럼명 ---
1. user
2. playtime
3. post_date
4. helpfulness
5. review
6. recommend
7. early_access_review
8. appid

--- df2 데이터 요약 정보 ---
<class 'pandas.DataFrame'>
RangeIndex: 31832390 entries, 0 to 31832389
Data columns (total 8 columns):
 #   Column               Dtype  
---  ------               -----  
 0   user                 str    
 1   playtime             float64
 2   post_date            str    
 3   helpfulness          int64  
 4   review               str    
 5   recommend            str    
 6   early_access_review  object 
 7   appid                str    
dtypes: float64(1), int64(1), object(1), str(5)
memory usage: 1.9+ GB

--- df2 데이터 미리보기 (상위 3줄) ---


,user,playtime,post_date,helpfulness,review,recommend,early_access_review,appid
0,Kutar'k,9.5,"May 16, 2021",13,"멸망한 인류가 간직한 강력한 힘의 해방.스플릿 불렛(Split Bullet), 벡토...",Recommended,NaN,1000000
1,DogelordCS,7.7,"July 5, 2021",5,"After beating this game, I would not recommend...",Not Recommended,NaN,1000000
2,점박이뱀,14.5,"May 16, 2021",8,형형색색 삐까번쩍 심플이즈베스트 도형디자인이 돋보이는 2D K-탄막슈팅 [어센션]입...,Recommended,NaN,1000000
3,haramey,8.1,"June 28, 2021",7,탄막 슈팅 / 어드벤처★★☆☆☆ (2/5)기대에 한참 못미친 비운의 게임★★★★☆ ...,Not Recommended,NaN,1000000
4,boymin,15.4,"May 18, 2021",7,전체적인 분위기는 와 시리즈를 섞어놓은 듯한 느낌입니다.(거기에 이제 슈팅 게임을...,Recommended,NaN,1000000
5,.Katjo,10.6,"May 28, 2021",3,Overall I liked it a lot but it does have a fe...,Recommended,NaN,1000000
6,zoronoa rollo,9.8,"May 15, 2021",7,지금까지 후기(플탐 2시간 정도)장점-도형 그래픽으로 심플하고 감각적임-BGM이 아...,Recommended,NaN,1000000
7,Fly252,9.1,"September 5, 2021",2,我一直很期待玩到银河恶魔城 + 俯视角弹幕射击游戏，这游戏满足了我的需求。所以，尽管这款游戏...,Recommended,NaN,1000000
8,Ho Most Unmost,6.2,"May 22, 2021",2,It's good.The top-down shooter and exploration...,Recommended,NaN,1000000
9,메트로,26.0,"November 4, 2021",3,재미는 있음근데 지도에서 내가 갔던 곳이 표시가 안되어 있어서 내가 어디를 가야 하...,Recommended,NaN,1000000


In [ ]:
# 1. post_date가 비어있는(결측치) 행의 개수와 비율 계산
missing_count = df2['post_date'].isnull().sum()
total_count = len(df2)
missing_ratio = (missing_count / total_count) * 100

print(f"전체 리뷰 수: {total_count:,}개")
print(f"post_date가 없는 데이터 수: {missing_count:,}개")
print(f"결측치 비율: {missing_ratio:.2f}%\n")

# 2. 실제로 post_date가 비어있는 데이터들만 모아서 확인해보기 (상위 5개)
if missing_count > 0:
    print("--- post_date가 없는 데이터 미리보기 ---")
    missing_data_df = df2[df2['post_date'].isnull()]
    display(missing_data_df.head())
else:
    print("🎉 post_date가 비어있는 데이터가 하나도 없습니다! (아주 깨끗한 상태)")

전체 리뷰 수: 31,832,390개
post_date가 없는 데이터 수: 0개
결측치 비율: 0.00%

🎉 post_date가 비어있는 데이터가 하나도 없습니다! (아주 깨끗한 상태)


In [ ]:
import pandas as pd

# 1. 'post_date' 칸에 날짜가 아닌 이상한 값(예: 숫자, 글자)이 들어간 경우 찾기
# 문자를 날짜로 강제 변환해보고, 변환에 실패해서 에러(NaT)가 나는 데이터만 걸러냅니다.
# (단, 원래 빈칸인 결측치는 제외)
parsed_dates = pd.to_datetime(df2['post_date'], errors='coerce')
invalid_date_mask = parsed_dates.isna() & df2['post_date'].notna()

# 2. 'helpfulness' (또는 평점 관련 칸)에 날짜 형태가 들어간 경우 찾기
# 보통 날짜에 쓰이는 기호('-', '/')나 연도(예: '2023', '2024')가 포함된 경우를 찾습니다.
date_in_wrong_col_mask = df2['helpfulness'].astype(str).str.contains(r'-|/|202[0-9]', na=False, regex=True)

# 3. 위 조건 중 하나라도 해당하는 '밀린/뒤바뀐 데이터' 추출
swapped_df = df2[invalid_date_mask | date_in_wrong_col_mask]

print(f"🚨 값이 옆으로 밀렸거나 뒤바뀐 것으로 의심되는 불량 데이터 수: {len(swapped_df):,}개\n")

if len(swapped_df) > 0:
    print("--- 🔍 의심되는 불량 데이터 미리보기 ---")
    # 전체 컬럼이 어떻게 밀렸는지 확인하기 위해 주요 컬럼들을 출력합니다.
    display(swapped_df[['appid', 'user','post_date', 'helpfulness', 'review', 'recommend']].head(15))
else:
    print("🎉 다행히 날짜와 평점이 뒤바뀐 데이터가 발견되지 않았습니다!")

🚨 값이 옆으로 밀렸거나 뒤바뀐 것으로 의심되는 불량 데이터 수: 15개

--- 🔍 의심되는 불량 데이터 미리보기 ---


,appid,user,post_date,helpfulness,review,recommend
657871,1059990,Hero_Vert,"September 21, 2022",2021,I am a classically trained trombonist (both te...,Recommended
1128328,1091500,cwot,"June 13, 2024",2024,"HelloI am a 54 yo father, probably one of the ...",Recommended
1317889,1097150,janeiro,"May 16, 2022",2021,"This will become F2P on June 21st, 2022",Recommended
1830310,1124300,小林子,"October 2, 2021",2022,我给个差评吧，希望他们可以看一下。人类这款游戏在国内的一些平台上的视频被下架了，具体的原因我...,Not Recommended
3549462,1180320,新建文件夹,"February 20, 2024",2026,兄弟们我是sb吗,Recommended
8150173,1326470,IsAmU,"February 23, 2023",2027,Po pierwszej godzinie gry jestem zakochany.Spo...,Recommended
9635636,1393350,大彻大悟,"November 12, 2023",2022,喜欢黄油的可以看看我下面的强烈推荐O(∩_∩)O哈哈~异世界猜拳勇者：Isekai Jank...,Recommended
15943760,1817230,Woid,"May 7, 2024",2027,RIP TangoNo more the evil within 3 and hi-fi r...,Recommended
17381306,1928980,P. Green,"February 20, 2024",2024,i will eat a spoon full of ketchup for every l...,Recommended
21208226,2357570,chwi :3,"December 6, 2023",2026,"Here I will leave the cat, friends who pass by...",Not Recommended


In [ ]:
import pandas as pd

# 1. playtime 칸의 값들을 강제로 숫자(Numeric)로 변환 시도
# 숫자로 바꿀 수 없는 값(날짜, 한글, 알파벳 등)은 변환에 실패하여 NaN(빈칸) 처리가 됩니다.
numeric_playtime = pd.to_numeric(df2['playtime'], errors='coerce')

# 2. 원래부터 빈칸이었던 데이터는 제외하고, 숫자로 변환에 실패한 '가짜 숫자(날짜 등)'만 걸러냅니다.
# (변환 후에는 NaN인데, 원본 데이터는 비어있지 않은 경우)
bad_playtime_mask = numeric_playtime.isna() & df2['playtime'].notna()

# 3. 불량 데이터만 추출
bad_playtime_df = df2[bad_playtime_mask]

print(f"🚨 playtime 칸에 숫자가 아닌 값(날짜/문자)이 들어간 데이터 수: {len(bad_playtime_df):,}개\n")

if len(bad_playtime_df) > 0:
    print("--- 🔍 의심되는 불량 데이터 미리보기 ---")
    # 어떻게 밀렸는지 파악하기 위해 주요 컬럼을 같이 보여줍니다.
    display(bad_playtime_df[['appid', 'user', 'playtime', 'post_date', 'helpfulness', 'review']].head(10))
else:
    print("🎉 playtime 칸에는 이상한 값이 없습니다!")

🚨 playtime 칸에 숫자가 아닌 값(날짜/문자)이 들어간 데이터 수: 0개

🎉 playtime 칸에는 이상한 값이 없습니다!
